## Training Arguments
- `TrainingArguments()`: customize training settings
- **See documentation for all parameters**
- Values depend on use, dataset, speed
- `output_dir`: output directory
- `eval_strategy`: when to evaluate "epoch", "steps", or "none"
- `num_train_epochs`: number of training epochs
- `learning_rate`: for optimizer
- `per_device_train_batch_size` and `per_device_eval_batch_size` define the batch size
- `weight_decay`: applied to the optimizer to avoid overfitting

In [1]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./finetuded',
    eval_strategy="epoch",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
)

## Trainer class
- `model`: the model to fine-tune
- `args`: the training arguments
- `train_dateset`: the data used for training
- `eval_dataset`: the data used for evaluation
- `tokenizer`: the tokenizer

Number of training loops: Dataset size, `num_train_epochs`, `per_device_train_batch_size` and `per_device_eval_batch_size`

In [2]:
from resources import get_resources

model, tokenizer, tokenized_training_data, tokenized_test_data = get_resources()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_training_data,
    eval_dataset=tokenized_test_data,
    tokenizer=tokenizer
)

trainer.train()

[+] Ingestão de dados iniciada (IMDB sharded)...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[+] Mapeando tokenizador sobre os lotes...


Map:   0%|          | 0/6250 [00:00<?, ? examples/s]

Map:   0%|          | 0/6250 [00:00<?, ? examples/s]

[+] Dutos alinhados. Prontos para o Trainer.


/tmp/ipykernel_21648/1596361348.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/phomint/PycharmProjects/DataCamp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.506200,0.522645
2,0.313500,0.502308
3,0.210500,0.759991


/home/phomint/PycharmProjects/DataCamp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/phomint/PycharmProjects/DataCamp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/phomint/PycharmProjects/DataCamp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=2346, training_loss=0.3268229150406235, metrics={'train_runtime': 3300.7309, 'train_samples_per_second': 5.681, 'train_steps_per_second': 0.711, 'total_flos': 616666536000000.0, 'train_loss': 0.3268229150406235, 'epoch': 3.0})

## Using the fine-tuned model

In [3]:
import torch

new_data = ['This is movie was disappointing!', 'This is the best movie ever!']

new_input = tokenizer(new_data, return_tensors="pt", padding=True, truncation=True, max_length=64)

with torch.no_grad():
    outputs = model(**new_input)

predicted_labels = torch.argmax(outputs.logits, dim=1).tolist()

label_map = {0: 'Negative', 1: 'Positive'}
for i, predicted_label in enumerate(predicted_labels):
    sentiment = label_map[predicted_label]
    print(f"Text: {new_data[i]} - Sentiment: {sentiment}")

Text: This is movie was disappointing! - Sentiment: Negative
Text: This is the best movie ever! - Sentiment: Positive


## Saving models and tokenizers

In [4]:
model.save_pretrained('my_finetuned_files')
tokenizer.save_pretrained('my_finetuned_files')

('my_finetuned_files/tokenizer_config.json',
 'my_finetuned_files/special_tokens_map.json',
 'my_finetuned_files/vocab.txt',
 'my_finetuned_files/added_tokens.json',
 'my_finetuned_files/tokenizer.json')

In [5]:
## loading a saved model
from transformers import AutoModelForSequenceClassification, AutoTokenizer


model = AutoModelForSequenceClassification.from_pretrained('my_finetuned_files')
tokenizer = AutoTokenizer.from_pretrained('my_finetuned_files')